# Notebook 05 — Search Optimization Service (SOS)
## Best Case vs. Worst Case

| | Best Design | Worst Design |
|---|---|---|
| **Query type** | Point lookup / substring on high-cardinality column | Broad range filter returning 60%+ of rows |
| **With SOS** | Sub-second across 500M rows | Zero benefit — SOS can't help |
| **Without SOS** | Full scan, 30+ seconds | Same as with SOS (SOS irrelevant) |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Worst Case — Point Lookup WITHOUT SOS

**Scenario**: Fraud investigator needs to find a specific transaction by ID.

In [ ]:
-- Point lookup without SOS — full scan of 500M rows for 1 record
SELECT *
FROM TRANSACTIONS
WHERE transaction_id = 'TXN-000250000000';

**Query Profile**: Note partitions scanned ≈ total partitions (full scan for 1 row!)

---
## Step 2: Enable Search Optimization

In [ ]:
-- Enable SOS on specific columns used for lookups
ALTER TABLE TRANSACTIONS ADD SEARCH OPTIMIZATION ON EQUALITY(transaction_id);
ALTER TABLE TRANSACTIONS ADD SEARCH OPTIMIZATION ON SUBSTRING(merchant_name);

-- Check SOS build progress
SHOW TABLES LIKE 'TRANSACTIONS';
-- Look at search_optimization_progress column

**Note**: SOS builds in the background. For a 500M-row table this may take several minutes.  
Wait for `search_optimization_progress = 100` before testing.

---
## Step 3: Best Case — Point Lookup WITH SOS

In [ ]:
-- Same query, now with SOS active — sub-second
SELECT *
FROM TRANSACTIONS
WHERE transaction_id = 'TXN-000250000000';

**Query Profile**: Partitions scanned drops to 1-3 (from thousands). Execution time: sub-second.

---
## Step 4: Best Case — Substring Search (Fraud Pattern)

**Scenario**: "Find all transactions at crypto exchanges" — substring search across 500M merchant names.

In [ ]:
-- Substring search WITH SOS — accelerated
SELECT transaction_date, account_id, amount, merchant_name
FROM TRANSACTIONS
WHERE merchant_name LIKE '%CRYPTO%'
  AND amount > 1000
ORDER BY amount DESC
LIMIT 20;

---
## Step 5: Worst Case — SOS on a Non-Selective Filter

**Scenario**: Broad filter that returns millions of rows — SOS provides zero benefit.

In [ ]:
-- WORST CASE: Low selectivity — returns ~60% of rows
-- SOS cannot help here (not a needle-in-haystack)
SELECT channel, COUNT(*), SUM(amount)
FROM TRANSACTIONS
WHERE amount > 100
GROUP BY 1;

**Query Profile**: Full scan regardless of SOS — the filter isn't selective enough.

---
## Step 6: Side-by-Side Metrics

In [ ]:
SELECT
    SUBSTR(query_text, 1, 60) AS query_preview,
    total_elapsed_time / 1000 AS elapsed_sec,
    query_id,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -15, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
WHERE start_time > DATEADD(minute, -15, CURRENT_TIMESTAMP())
WHERE query_text ILIKE '%TRANSACTIONS%WHERE%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 4;

---
## Step 7: Cost Awareness

In [ ]:
-- What does SOS cost for this table?
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('TRANSACTIONS', 'EQUALITY(transaction_id)');

---
## Key Takeaways

| | Best Design | Worst Design |
|---|---|---|
| **Use SOS for** | `transaction_id` lookup, `merchant_name LIKE '%..%'`, `ssn_hash` equality | `amount > 100` (returns millions of rows) |
| **Benefit** | Full scan → 1-3 partitions (1000x fewer) | Zero — SOS only helps selective queries |
| **Anti-pattern** | — | Enabling SOS on every column "just in case" (high maintenance cost) |

**Banking use cases for SOS**:
- Fraud: specific transaction lookup, merchant pattern search
- Compliance: find customer by SSN hash
- Customer service: locate specific account activity

**Next** → [Notebook 06 — Query Tuning Best Practices](./Notebook_06_Query_Tuning.ipynb)